In [ ]:
!pip install fastapi
!pip install unicorn

In [ ]:
import json
import time
import httpx
import tiktoken
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse

UPSTREAM_URL = "http://localhost:8080/v1/chat/completions"

# pricing
INPUT_COST = 25 / 1_000_000
OUTPUT_COST = 75 / 1_000_000

app = FastAPI()

enc = tiktoken.get_encoding("cl100k_base")

def count_tokens(text: str) -> int:
    return len(enc.encode(text))


@app.post("/v1/chat/completions")
async def proxy(request: Request):
    body = await request.json()

    messages = body.get("messages", [])
    prompt_text = "\n".join([m.get("content", "") for m in messages])

    input_tokens = count_tokens(prompt_text)
    output_tokens = 0
    start_time = time.time()

    cost = 0.0

    async def stream():
        nonlocal output_tokens, cost

        async with httpx.AsyncClient(timeout=None) as client:
            async with client.stream(
                "POST",
                UPSTREAM_URL,
                json=body
            ) as resp:

                async for line in resp.aiter_lines():
                    if not line:
                        continue

                    # pass-through normal SSE
                    yield line + "\n"

                    # parse streamed tokens
                    if line.startswith("data: "):
                        data = line[6:]

                        if data.strip() == "[DONE]":
                            continue

                        try:
                            chunk = json.loads(data)
                            delta = chunk["choices"][0].get("delta", {})
                            content = delta.get("content")

                            if content:
                                output_tokens += count_tokens(content)

                        except Exception:
                            pass

        # final cost calculation
        cost = (input_tokens * INPUT_COST) + (output_tokens * OUTPUT_COST)

        # 🔥 IMPORTANT: LiteLLM-friendly usage event
        usage_event = {
            "type": "usage",
            "usage": {
                "prompt_tokens": input_tokens,
                "completion_tokens": output_tokens,
                "total_tokens": input_tokens + output_tokens,
                "cost_usd": round(cost, 8),
                "latency_sec": round(time.time() - start_time, 3)
            }
        }

        yield f"data: {json.dumps(usage_event)}\n\n"
        yield "data: [DONE]\n\n"

    return StreamingResponse(stream(), media_type="text/event-stream")

INFO:     127.0.0.1:61010 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO:     127.0.0.1:61015 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO:     127.0.0.1:61018 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO:     127.0.0.1:61023 - "OPTIONS /chat/completions HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:61024 - "OPTIONS /chat/completions HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:61024 - "OPTIONS /chat/completions HTTP/1.1" 404 Not Found


: 

In [ ]:
import uvicorn
import threading

def run():
    uvicorn.run(app, host="0.0.0.0", port=9000, log_level="info")

thread = threading.Thread(target=run, daemon=True)
thread.start()

INFO:     Started server process [81947]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:9000 (Press CTRL+C to quit)


In [ ]:
import httpx

r = httpx.post(
    "http://127.0.0.1:9000/v1/chat/completions",
    json={
        "messages": [{"role": "user", "content": "hello world"}],
        "stream": True
    },
    timeout=None
)

for line in r.text.splitlines():
    print(line)

INFO:     127.0.0.1:60677 - "POST /v1/chat/completions HTTP/1.1" 200 OK
data: {"choices":[{"finish_reason":null,"index":0,"delta":{"role":"assistant","content":null}}],"created":1778680449,"id":"chatcmpl-jnG0OwajLohR54dV4uZEjKBSfzBvKhYW","model":"unsloth/gemma-4-E4B-it-GGUF:Q8_0","system_fingerprint":"b9121-89730c8d2","object":"chat.completion.chunk"}
data: {"choices":[{"finish_reason":null,"index":0,"delta":{"content":"Hello"}}],"created":1778680449,"id":"chatcmpl-jnG0OwajLohR54dV4uZEjKBSfzBvKhYW","model":"unsloth/gemma-4-E4B-it-GGUF:Q8_0","system_fingerprint":"b9121-89730c8d2","object":"chat.completion.chunk"}
data: {"choices":[{"finish_reason":null,"index":0,"delta":{"content":" there"}}],"created":1778680449,"id":"chatcmpl-jnG0OwajLohR54dV4uZEjKBSfzBvKhYW","model":"unsloth/gemma-4-E4B-it-GGUF:Q8_0","system_fingerprint":"b9121-89730c8d2","object":"chat.completion.chunk"}
data: {"choices":[{"finish_reason":null,"index":0,"delta":{"content":"!"}}],"created":1778680449,"id":"chatcmpl-j

INFO:     127.0.0.1:60852 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO:     127.0.0.1:60944 - "POST /v1/chat/completions HTTP/1.1" 200 OK
